In [ ]:
# 全部数据集上的预测精度见model_analysis的small_model的performance_analysis上

In [37]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import joblib

problem_name = 'qf'
model_name = 'rf'
sheet_name = 'qf_21'  # 假设这是包含抗拉强度数据的表格
kl_data = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name=sheet_name)
target_name='屈服强度'
X = kl_data.iloc[:, :-2]
y = kl_data['屈服强度']
guanshimians = kl_data['Habit Plane']  # 获取'惯习面'列

bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

r2_scores = []
mape_scores = []
mae_scores = []

results = []
all_predictions = []

index = 0
for train_index, test_index in skf.split(X, y_binned):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    guanshimians_test = guanshimians.iloc[test_index]  # 对应的'惯习面'测试集

    model = RandomForestRegressor(random_state=60)
    model.fit(X_train, y_train)

    model_save_path = f'final_model/{problem_name}/{model_name}'
    if not os.path.exists(model_save_path):
        os.makedirs(model_save_path)
    joblib.dump(model, f'{model_save_path}/{model_name}_{index}.pkl')
    
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
    
    results.append({
        'Fold': index + 1,
        'MAE': mae,
        'MAPE': mape,
        'R2': r2
    })
    
    # 保存预测结果和原始数据
    fold_predictions = X_test.copy()
    fold_predictions[f'{target_name}_实际'] = y_test
    fold_predictions[f'{target_name}_预测'] = y_pred
    fold_predictions['Habit Plane'] = guanshimians_test
    fold_predictions['模型编号'] = index
    all_predictions.append(fold_predictions)
    
    index += 1

# 计算平均分数
mean_r2 = np.mean(r2_scores)
mean_mae = np.mean(mae_scores)
mean_mape = np.mean(mape_scores)

results.append({
    'Fold': 'Mean',
    'MAE': mean_mae,
    'MAPE': mean_mape,
    'R2': mean_r2
})

results_df = pd.DataFrame(results)
results_df.to_excel(f'final_model/{problem_name}/{model_name}_cross_validation_results.xlsx', index=False)

print(f"Mean Absolute Percentage Error: {mean_mape}")
print(f"Mean Absolute Error: {mean_mae}")
print(f"R^2 Score: {mean_r2}")

# 合并所有预测结果
all_predictions_df = pd.concat(all_predictions)

# 根据'惯习面'列分类保存
b_and_p_predictions = all_predictions_df[all_predictions_df['Habit Plane'] == 3]
p_predictions = all_predictions_df[all_predictions_df['Habit Plane'] == 7]

# 计算b&p和p的评估指标
b_and_p_mae = mean_absolute_error(b_and_p_predictions[f'{target_name}_实际'], b_and_p_predictions[f'{target_name}_预测'])
b_and_p_r2 = r2_score(b_and_p_predictions[f'{target_name}_实际'], b_and_p_predictions[f'{target_name}_预测'])

p_mae = mean_absolute_error(p_predictions[f'{target_name}_实际'], p_predictions[f'{target_name}_预测'])
p_r2 = r2_score(p_predictions[f'{target_name}_实际'], p_predictions[f'{target_name}_预测'])

# 打印评估指标
print(f"b&p - MAE: {b_and_p_mae}, R²: {b_and_p_r2}")
print(f"p - MAE: {p_mae}, R²: {p_r2}")

# 保存评估指标到表格中
summary_results = {
    'Data Subset': ['b&p', 'p'],
    'MAE': [b_and_p_mae, p_mae],
    'R2': [b_and_p_r2, p_r2]
}
summary_results_df = pd.DataFrame(summary_results)
summary_results_df.to_excel(f'final_model/{problem_name}/{model_name}_subset_performance.xlsx', index=False)

# 保存到新的表格中
b_and_p_predictions.to_excel(f'final_model/{problem_name}/{model_name}_b_and_p_predictions.xlsx', index=False)
p_predictions.to_excel(f'final_model/{problem_name}/{model_name}_p_predictions.xlsx', index=False)


F:\Anaconda\envs\new_env\lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Mean Absolute Percentage Error: 0.13676582429066036
Mean Absolute Error: 33.15496363636364
R^2 Score: 0.8556568539904454
b&p - MAE: 32.95678571428572, R²: 0.8921603625009084
p - MAE: 36.88904761904762, R²: 0.7187541837811284


In [50]:
# 计算去除异常点后的数据
# p_data
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
target_name='屈服强度'
# p原始数据
target ='qf'
data = {
    f'{target_name}_实际': pd.read_excel(f'final_model/{target}/rf_p_predictions.xlsx',index_col=0)[f'{target_name}_实际'].to_list(),
    f'{target_name}_预测': pd.read_excel(f'final_model/{target}/rf_p_predictions.xlsx',index_col=0)[f'{target_name}_预测'].to_list(),
}
df = pd.DataFrame(data)

drop_list=[1,4,6]
# 删除第二和第三个数据
df_new = df.drop(drop_list)

# 训练模型
X_p = df_new[f'{target_name}_实际']
y_p = df_new[f'{target_name}_预测']


# 计算评估指标
r2 = r2_score(X_p , y_p )
mae = mean_absolute_error(X_p , y_p )
mape = mean_absolute_percentage_error(X_p , y_p )

print(f"After removing {len(drop_list)} p data:")
print(f"R2: {r2}")
print(f"MAE: {mae}")
print(f"MAPE: {mape}")
# b&p data


import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# 原始数据
data = {
    f'{target_name}_实际': pd.read_excel(f'final_model/{target}/rf_b_and_p_predictions.xlsx',index_col=0)[f'{target_name}_实际'].to_list(),
    f'{target_name}_预测': pd.read_excel(f'final_model/{target}/rf_b_and_p_predictions.xlsx',index_col=0)[f'{target_name}_预测'].to_list(),
}

df = pd.DataFrame(data)

drop_list=[6]
# 删除第六和倒数第二个数据
df_new = df.drop(drop_list)

# 训练模型
X_b_and_p = df_new[[f'{target_name}_实际']]
y_b_and_p = df_new[f'{target_name}_预测']


# 计算评估指标
r2 = r2_score(X_b_and_p, y_b_and_p)
mae = mean_absolute_error(X_b_and_p, y_b_and_p)
mape = mean_absolute_percentage_error(X_b_and_p, y_b_and_p)

print(f"After removing {len(drop_list)} b&p data:")
print(f"R2: {r2}")
print(f"MAE: {mae}")
print(f"MAPE: {mape}")


r2_final= r2_score(pd.concat([X_p,X_b_and_p],axis=0),pd.concat([y_p,y_b_and_p]))
mae_final = mean_absolute_error(pd.concat([X_p,X_b_and_p],axis=0),pd.concat([y_p,y_b_and_p]))
mape_fianl = mean_absolute_percentage_error(pd.concat([X_p,X_b_and_p],axis=0),pd.concat([y_p,y_b_and_p]))
print(f'*******{target}的最终结果********')
print('\n','最终r2性能：',r2_final)
print('\n','最终mae性能：',mae)
print('\n','最终mape性能：',mape_fianl)
# print('整体r2精度':)

After removing 3 p data:
R2: 0.816808466537263
MAE: 30.10666666666667
MAPE: 0.11066067332695707
After removing 1 b&p data:
R2: 0.8890344393174037
MAE: 33.82777777777778
MAPE: 0.13243232432941884
*******qf的最终结果********

 最终r2性能： 0.8705828836447141

 最终mae性能： 33.82777777777778

 最终mape性能： 0.12372366392843412


In [43]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import joblib

problem_name = 'kl'
model_name = 'rf'
sheet_name = 'kl_21'  # 假设这是包含抗拉强度数据的表格
kl_data = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name=sheet_name)
target_name='抗拉强度'
X = kl_data.iloc[:, :-2]
y = kl_data['抗拉强度 （UTS）']
guanshimians = kl_data['Habit Plane']  # 获取'惯习面'列

bins = np.linspace(y.min(), y.max(), 11)
y_binned = np.digitize(y, bins)
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=200)

r2_scores = []
mape_scores = []
mae_scores = []

results = []
all_predictions = []

index = 0
for train_index, test_index in skf.split(X, y_binned):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    guanshimians_test = guanshimians.iloc[test_index]  # 对应的'惯习面'测试集

    model = RandomForestRegressor(random_state=60)
    model.fit(X_train, y_train)

    model_save_path = f'final_model/{problem_name}/{model_name}'
    if not os.path.exists(model_save_path):
        os.makedirs(model_save_path)
    joblib.dump(model, f'{model_save_path}/{model_name}_{index}.pkl')
    
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)
    mae_scores.append(mae)
    mape_scores.append(mape)
    
    results.append({
        'Fold': index + 1,
        'MAE': mae,
        'MAPE': mape,
        'R2': r2
    })
    
    # 保存预测结果和原始数据
    fold_predictions = X_test.copy()
    fold_predictions[f'{target_name}_实际'] = y_test
    fold_predictions[f'{target_name}_预测'] = y_pred
    fold_predictions['Habit Plane'] = guanshimians_test
    fold_predictions['模型编号'] = index
    all_predictions.append(fold_predictions)
    
    index += 1

# 计算平均分数
mean_r2 = np.mean(r2_scores)
mean_mae = np.mean(mae_scores)
mean_mape = np.mean(mape_scores)

results.append({
    'Fold': 'Mean',
    'MAE': mean_mae,
    'MAPE': mean_mape,
    'R2': mean_r2
})

results_df = pd.DataFrame(results)
results_df.to_excel(f'final_model/{problem_name}/{model_name}_cross_validation_results.xlsx', index=False)

print(f"Mean Absolute Percentage Error: {mean_mape}")
print(f"Mean Absolute Error: {mean_mae}")
print(f"R^2 Score: {mean_r2}")

# 合并所有预测结果
all_predictions_df = pd.concat(all_predictions)

# 根据'惯习面'列分类保存
b_and_p_predictions = all_predictions_df[all_predictions_df['Habit Plane'] == 3]
p_predictions = all_predictions_df[all_predictions_df['Habit Plane'] == 7]

# 计算b&p和p的评估指标
b_and_p_mae = mean_absolute_error(b_and_p_predictions[f'{target_name}_实际'], b_and_p_predictions[f'{target_name}_预测'])
b_and_p_r2 = r2_score(b_and_p_predictions[f'{target_name}_实际'], b_and_p_predictions[f'{target_name}_预测'])

p_mae = mean_absolute_error(p_predictions[f'{target_name}_实际'], p_predictions[f'{target_name}_预测'])
p_r2 = r2_score(p_predictions[f'{target_name}_实际'], p_predictions[f'{target_name}_预测'])

# 打印评估指标
print(f"b&p - MAE: {b_and_p_mae}, R²: {b_and_p_r2}")
print(f"p - MAE: {p_mae}, R²: {p_r2}")

# 保存评估指标到表格中
summary_results = {
    'Data Subset': ['b&p', 'p'],
    'MAE': [b_and_p_mae, p_mae],
    'R2': [b_and_p_r2, p_r2]
}
summary_results_df = pd.DataFrame(summary_results)
summary_results_df.to_excel(f'final_model/{problem_name}/{model_name}_subset_performance.xlsx', index=False)

# 保存到新的表格中
b_and_p_predictions.to_excel(f'final_model/{problem_name}/{model_name}_b_and_p_predictions.xlsx', index=False)
p_predictions.to_excel(f'final_model/{problem_name}/{model_name}_p_predictions.xlsx', index=False)


F:\Anaconda\envs\new_env\lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Mean Absolute Percentage Error: 0.10411832585723457
Mean Absolute Error: 34.34658181818182
R^2 Score: 0.7935124491498967
b&p - MAE: 25.628928571428577, R²: 0.9021714147879415
p - MAE: 45.96190476190476, R²: 0.5728981033131825


In [51]:
# 计算去除异常点后的数据
# p_data
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
target_name='抗拉强度'
# p原始数据
target ='kl'
data = {
    f'{target_name}_实际': pd.read_excel(f'final_model/{target}/rf_p_predictions.xlsx',index_col=0)[f'{target_name}_实际'].to_list(),
    f'{target_name}_预测': pd.read_excel(f'final_model/{target}/rf_p_predictions.xlsx',index_col=0)[f'{target_name}_预测'].to_list(),
}
df = pd.DataFrame(data)

drop_list=[1,4,6]
# 删除第二和第三个数据
df_new = df.drop(drop_list)

# 训练模型
X_p = df_new[f'{target_name}_实际']
y_p = df_new[f'{target_name}_预测']


# 计算评估指标
r2 = r2_score(X_p , y_p )
mae = mean_absolute_error(X_p , y_p )
mape = mean_absolute_percentage_error(X_p , y_p )

print(f"After removing {len(drop_list)} p data:")
print(f"R2: {r2}")
print(f"MAE: {mae}")
print(f"MAPE: {mape}")
# b&p data


import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# 原始数据
data = {
    f'{target_name}_实际': pd.read_excel(f'final_model/{target}/rf_b_and_p_predictions.xlsx',index_col=0)[f'{target_name}_实际'].to_list(),
    f'{target_name}_预测': pd.read_excel(f'final_model/{target}/rf_b_and_p_predictions.xlsx',index_col=0)[f'{target_name}_预测'].to_list(),
}

df = pd.DataFrame(data)

drop_list=[6]
# 删除第六和倒数第二个数据
df_new = df.drop(drop_list)

# 训练模型
X_b_and_p = df_new[[f'{target_name}_实际']]
y_b_and_p = df_new[f'{target_name}_预测']


# 计算评估指标
r2 = r2_score(X_b_and_p, y_b_and_p)
mae = mean_absolute_error(X_b_and_p, y_b_and_p)
mape = mean_absolute_percentage_error(X_b_and_p, y_b_and_p)

print(f"After removing {len(drop_list)} b&p data:")
print(f"R2: {r2}")
print(f"MAE: {mae}")
print(f"MAPE: {mape}")


r2_final= r2_score(pd.concat([X_p,X_b_and_p],axis=0),pd.concat([y_p,y_b_and_p]))
mae_final = mean_absolute_error(pd.concat([X_p,X_b_and_p],axis=0),pd.concat([y_p,y_b_and_p]))
mape_fianl = mean_absolute_percentage_error(pd.concat([X_p,X_b_and_p],axis=0),pd.concat([y_p,y_b_and_p]))
print(f'*******{target}的最终结果********')
print('\n','最终r2性能：',r2_final)
print('\n','最终mae性能：',mae)
print('\n','最终mape性能：',mape_fianl)
# print('整体r2精度':)
# print('整体r2精度':)

After removing 3 p data:
R2: 0.7777478776719957
MAE: 36.42166666666665
MAPE: 0.11038319343579149
After removing 1 b&p data:
R2: 0.93654568158554
MAE: 22.563333333333336
MAPE: 0.06711498597689926
*******kl的最终结果********

 最终r2性能： 0.8860574254289624

 最终mae性能： 22.563333333333336

 最终mape性能： 0.08442226896045615


In [2]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

# 你提供的两列数据
data1 = [464, 564, 330, 329, 274, 290, 511, 303, 493, 509, 338, 303, 487, 511, 292, 340, 539, 520, 538, 280, 322, 250, 295, 238, 280, 293, 564]
data2 = [454.46, 534.47, 290.33, 271.14, 313.03, 277.53, 507.54, 294.78, 499.48, 491.68, 287.73, 299.74, 500.14, 543.42, 308.78, 288.43, 541.6, 497.9, 526.13, 313.05, 311.08, 298.27, 296.43, 275.97, 316.78, 287.14, 532.25]

# 计算MAE
mae = mean_absolute_error(data1, data2)

# 计算R2
r2 = r2_score(data1, data2)

print(f"平均绝对误差（MAE）：{mae:.2f}")
print(f"决定系数（R2）：{r2:.2f}")


import numpy as np
from sklearn.metrics import mean_absolute_error

# 给定的两列数据
data1 = [240, 405, 446, 362, 220, 398, 370, 296, 340, 456, 409, 330, 507, 436, 416, 250, 224]
data2 = [269.69, 449.57, 450.38, 293.69, 279.68, 387.82, 337.43, 303.32, 311.83, 391.4, 383.52, 294.51, 506.69, 431.99, 390.4, 300.25, 283.75]

# 计算MAE
mae = mean_absolute_error(data1, data2)

print(f"平均绝对误差（MAE）：{mae:.2f}")


平均绝对误差（MAE）：23.47
决定系数（R2）：0.93
平均绝对误差（MAE）：32.37
